# 03 — Embedding & Index Building

**Pipeline position:** After chunking (02), before retrieval (04).

**Purpose:** Encode chunks into dense vectors (BGE-M3, 1024-dim) stored in Chroma,
and build a BM25 sparse index for hybrid retrieval.

**Learning objectives:**
- Understand dense vs sparse representations
- Build a Chroma PersistentClient index with cosine similarity
- Build a BM25 index with English NLTK tokenization
- Verify index integrity and run a basic similarity search

## Imports

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import matplotlib.pyplot as plt
from src import constant
from src.client.mongodb_config import MongoConfig
from src.embedder import Embedder, BGEEmbedder, ChromaStore, english_tokenize

## Load chunks from MongoDB

We retrieve all child chunks that were stored during the chunking step.

In [ ]:
collection = MongoConfig.get_collection(constant.mongo_collection)
chunks = list(collection.find({}, {'_id': 0}))
print(f'Total chunks in MongoDB: {len(chunks)}')
if chunks:
    print(f'Sample keys: {list(chunks[0].keys())}')
    print(f'Sample text: {chunks[0].get("page_content", "")[:200]}...')

## Encode a sample with BGE-M3

BGE-M3 produces 1024-dimensional dense vectors. Let's encode a single chunk and inspect.

In [ ]:
embedder = BGEEmbedder()
sample_text = chunks[0]['page_content'] if chunks else 'T cells are activated by antigen presentation.'
vec = embedder.encode_query(sample_text)
print(f'Vector type: {type(vec)}')
print(f'Vector dimension: {len(vec)}')
print(f'First 10 values: {vec[:10]}')
print(f'L2 norm: {np.linalg.norm(vec):.4f}')

## Build Chroma index

Chroma PersistentClient stores embeddings on disk with HNSW for approximate nearest neighbor search.

In [ ]:
chroma_store = ChromaStore()
print(f'Chroma collection: {constant.chroma_collection}')
print(f'Current count: {chroma_store.collection.count()}')

## Build BM25 index

BM25 uses English tokenization (NLTK word_tokenize + stopword removal) instead of jieba.

In [ ]:
# Demonstrate English tokenization
sample = 'T cell activation requires TCR recognition of peptide-MHC complexes.'
tokens = english_tokenize(sample)
print(f'Original: {sample}')
print(f'Tokens:   {tokens}')
print(f'Token count: {len(tokens)}')

## Index report

After building, the Embedder writes a report with chunk counts, dimensions, and disk size.

In [ ]:
from pathlib import Path
report_path = Path(constant.diagnostics_dir) / 'index_report.txt'
if report_path.exists():
    print(report_path.read_text(encoding='utf-8'))
else:
    print('No index report yet. Run: python build_index.py')

## Test similarity search

Query the Chroma index directly to verify it returns relevant results.

In [ ]:
query = 'What activates T cells?'
query_vec = embedder.encode_query(query)

results = chroma_store.collection.query(
    query_embeddings=[query_vec],
    n_results=5,
    include=['documents', 'distances']
)

print(f'Query: {query}\n')
for i, (text, dist) in enumerate(zip(results['documents'][0], results['distances'][0]), 1):
    print(f'[{i}] (distance={dist:.4f}) {text[:120]}...\n')

## Summary

**Outputs produced:**
- Chroma persistent index at `outputs/vectorstore/`
- BM25 pickle at `outputs/vectorstore/bm25_index.pkl`
- Index report at `outputs/diagnostics/index_report.txt`

**Next:** `04_retriever.ipynb` — Hybrid retrieval with BM25 + Dense + RRF fusion.